<h1 style="font-family:verdana;"><center>🗂️OTTO: LightGBM brief sample code with cudf</center></h1>

# References
**Great notebooks! Thank you🙇**
- https://www.kaggle.com/code/cdeotte/candidate-rerank-model-lb-0-575
- https://www.kaggle.com/code/greenwolf/lightgbm-fast-recall-20/notebook

**My other notebook📒:**
- 🥉https://www.kaggle.com/code/mizuny/otto-easy-to-use-co-visitation-matrix-function
- https://www.kaggle.com/code/mizuny/otto-eda-notebook

# In this notebook
- LightGBM ranking training & validation
- Fast reading data with cudf
- Brief commentary for LightGBM parameter `group`

# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import glob, gc
import cudf
print('We will use RAPIDS version',cudf.__version__)
import lightgbm as lgb

<div class="alert alert-block alert-info" style="font-size:14px; font-family:verdana;">
    📌For description of loading data, read following notebook: <a href="https://www.kaggle.com/code/cdeotte/candidate-rerank-model-lb-0-575" target="_blank" style="color:#ff7f7f;">Candidate ReRank Model - [LB 0.575] by cdeotte</a>
</div>

In [ ]:
%%time
# CACHE FUNCTIONS
def read_file(f):
    return cudf.DataFrame( data_cache[f] )

def read_file_to_cache(f):
    df = pd.read_parquet(f)
    df.ts = (df.ts/1000).astype('int32')
    df['type'] = df['type'].map(type_labels).astype('int8')
    return df

# CACHE THE DATA ON CPU BEFORE PROCESSING ON GPU
data_cache = {}
type_labels = {'clicks':0, 'carts':1, 'orders':2}
files = glob.glob('../input/otto-chunk-data-inparquet-format/*_parquet/*')
for f in files: data_cache[f] = read_file_to_cache(f)

# CHUNK PARAMETERS
READ_CT = 5
OUTER_CHUNK_NUM = 100
CHUNK_NUM = int( np.ceil( len(files) / OUTER_CHUNK_NUM ))
print(f'We will process {len(files)} files, in groups of {READ_CT} and chunks of {CHUNK_NUM}.')

# Loading Data

In [ ]:
drop_th_sess_num = 30

for outer_chk_id in range(OUTER_CHUNK_NUM):
    start_chk_id = outer_chk_id * CHUNK_NUM
    end_chk_id = min( (outer_chk_id+1)*CHUNK_NUM, len(files) )

    for inner_chk_id in range(start_chk_id, end_chk_id, READ_CT):
        df = [read_file(files[inner_chk_id])]
        # read inner chunk files
        for i in range(1, READ_CT): 
            if (inner_chk_id+i) < end_chk_id:
                df.append( read_file(files[(inner_chk_id+i)]) )

        df = cudf.concat(df,ignore_index=True,axis=0)

        # USE TAIL OF SESSION
        df = df.reset_index(drop=True)
        df['n'] = df.groupby('session').cumcount()
        # ** SPECIFY BY ARG **
        df = df.loc[df.n < drop_th_sess_num].drop('n',axis=1)
        
        # COMBINE INNER CHUNKS
        if inner_chk_id == start_chk_id:
            df_concat = df
        else:
            df_concat = cudf.concat([df_concat, df], ignore_index=True, axis=0)
        print(inner_chk_id,', ',end='')

    del df
    gc.collect()
    break # if use all data. please comment out

In [ ]:
df_concat = df_concat.sort_values('ts',ascending=False)
df_concat = df_concat.reset_index(drop=True)
df_concat

In [ ]:
# identify boundary index between train and val
# 80% for train, 20% for val
fold_size = len(df_concat) // 5

# Create train / val dataset

In [ ]:
val_df = df_concat.iloc[:fold_size]
val_df = val_df.reset_index(drop=True)

train_df = df_concat.iloc[fold_size:]
train_df = train_df.reset_index(drop=True)

del df_concat
gc.collect()

In [ ]:
train_query = train_df.groupby(by='session', sort=True).size()
val_query = val_df.groupby(by='session', sort=True).size()

- **`group` parameter for LightGBM**
> group (list, numpy 1-D array, pandas Series or None, optional (default=None)) – Group/query data. Only used in the learning-to-rank task. sum(group) = n_samples. For example, if you have a 100-document dataset with group = [10, 20, 40, 10, 10, 10], that means that you have 6 groups, where the first 10 records are in the first group, records 11-30 are in the second group, records 31-70 are in the third group, etc.
by [LightGBM documentation](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Dataset.html)

In [ ]:
print(f'train size: {len(train_df)}, ')
print(f'val size: {len(val_df)}')

In [ ]:
target_col = 'type'
train_dataset = lgb.Dataset(
    train_df.drop([target_col], axis=1).values.get(),
    train_df[target_col].values.astype(int).get(),
    group=train_query.values.get())

val_dataset = lgb.Dataset(
    val_df.drop([target_col], axis=1).values.get(),
    val_df[target_col].values.astype(int).get(),
    group=val_query.values.get())

In [ ]:
%%time

params = {
    'objective': 'lambdarank',
#     'metric': '"None"', # if use custom metric, specify "None"
    'learning_rate': 0.01,
    'boosting_type': 'gbdt',
#     'label_gain': label_gain
}

model = lgb.train(
    params,
    train_dataset,
    valid_sets=val_dataset,
#     feval=custom_metric_func,
    callbacks=[lgb.log_evaluation(5)],
)

In [ ]:
y_pred = model.predict(val_df.drop(['type'], axis=1).values.get())